# Private audio to structured meeting notes with FunASR and Claude

This cookbook transcribes an audio file with a self-hosted [FunASR](https://github.com/modelscope/FunASR) server, then asks Claude to turn the transcript into concise meeting notes.

The audio stays on the machine running FunASR. The transcript is sent to the Claude API when `ANTHROPIC_API_KEY` is configured. Review the transcript before sending it if it contains sensitive information.


## 1. Start the local transcription server

In a terminal, install the current FunASR package and bind the OpenAI-compatible server to the IPv4 loopback interface:

```bash
python -m pip install -U "funasr>=1.3.26"
funasr-server --model sensevoice --device cpu --host 127.0.0.1 --port 8000
```

Use `--device cuda` on a compatible GPU host. Keep the service on loopback for this local recipe; use authentication, TLS, request limits, and network controls before exposing it beyond the machine.


## 2. Configure the notebook

The repository environment already includes `anthropic` and `requests`. For a standalone environment:

```bash
python -m pip install -U "anthropic>=0.109.0" "requests>=2.32.5" notebook
export ANTHROPIC_API_KEY="your-key"
export AUDIO_PATH="/absolute/path/to/meeting.wav"
```

If `AUDIO_PATH` or `ANTHROPIC_API_KEY` is not set, the notebook uses deterministic demo content so every cell can still run offline.


In [1]:
import mimetypes
import os
from pathlib import Path

import requests
from anthropic import Anthropic

FUNASR_BASE_URL = os.getenv("FUNASR_BASE_URL", "http://127.0.0.1:8000").rstrip("/")
AUDIO_PATH = Path(os.getenv("AUDIO_PATH", "meeting.wav")).expanduser()
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-haiku-4-5")
REQUEST_TIMEOUT_SECONDS = 300

print(f"FunASR endpoint: {FUNASR_BASE_URL}")
print(f"Audio path: {AUDIO_PATH}")
print(f"Claude model: {CLAUDE_MODEL}")

FunASR endpoint: http://127.0.0.1:8000
Audio path: meeting.wav
Claude model: claude-haiku-4-5


## 3. Transcribe through FunASR

FunASR implements the familiar multipart `POST /v1/audio/transcriptions` contract. The helper validates the local file, applies a finite timeout, surfaces HTTP errors, and rejects an empty transcript.


In [2]:
def transcribe_audio(
    audio_path: Path,
    *,
    base_url: str = FUNASR_BASE_URL,
    model: str = "sensevoice",
    timeout: int = REQUEST_TIMEOUT_SECONDS,
) -> str:
    """Transcribe one local audio file with the FunASR OpenAI-compatible API."""
    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    content_type = mimetypes.guess_type(audio_path.name)[0] or "application/octet-stream"
    with audio_path.open("rb") as audio_file:
        response = requests.post(
            f"{base_url}/v1/audio/transcriptions",
            data={"model": model},
            files={"file": (audio_path.name, audio_file, content_type)},
            timeout=timeout,
        )

    response.raise_for_status()
    payload = response.json()
    transcript = str(payload.get("text", "")).strip()
    if not transcript:
        raise RuntimeError("FunASR returned an empty transcript")
    return transcript

In [3]:
DEMO_TRANSCRIPT = """Maya: We will ship the multilingual search beta on Friday.
Chen: I will finish the relevance evaluation by Wednesday.
Maya: The open question is whether Japanese indexing stays in the beta.
Luis: I will confirm the rollout owner and publish the dashboard link tomorrow.
"""

if AUDIO_PATH.is_file():
    transcript = transcribe_audio(AUDIO_PATH)
    transcript_source = f"Live FunASR transcript from {AUDIO_PATH}"
else:
    transcript = DEMO_TRANSCRIPT
    transcript_source = "Offline demo transcript (set AUDIO_PATH to call FunASR)"

print(transcript_source)
print(transcript)

Offline demo transcript (set AUDIO_PATH to call FunASR)
Maya: We will ship the multilingual search beta on Friday.
Chen: I will finish the relevance evaluation by Wednesday.
Maya: The open question is whether Japanese indexing stays in the beta.
Luis: I will confirm the rollout owner and publish the dashboard link tomorrow.



## 4. Ask Claude for structured notes

The prompt requires four stable Markdown sections. It also tells Claude not to invent owners or deadlines, which matters when the transcript is incomplete or ASR confidence is low.


In [4]:
MEETING_NOTES_PROMPT = """Turn the meeting transcript into concise Markdown notes with exactly these headings:

## Summary
## Decisions
## Action items
## Open questions

For each action item, include the owner and deadline only when they are explicit in the transcript.
Do not invent facts. Preserve uncertainty and call out unclear names or dates.
"""


def create_meeting_notes(
    transcript: str,
    *,
    client: Anthropic | None = None,
    model: str = CLAUDE_MODEL,
) -> str:
    """Create structured meeting notes from a non-empty transcript."""
    if not transcript.strip():
        raise ValueError("Transcript must not be empty")

    anthropic_client = client or Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    message = anthropic_client.messages.create(
        model=model,
        max_tokens=1200,
        system=MEETING_NOTES_PROMPT,
        messages=[{"role": "user", "content": transcript}],
    )
    text_blocks = [
        block.text for block in message.content if getattr(block, "type", None) == "text"
    ]
    if not text_blocks:
        raise RuntimeError("Claude returned no text content")
    return "\n\n".join(text_blocks)

In [5]:
DEMO_NOTES = """## Summary
The team plans to ship the multilingual search beta on Friday.

## Decisions
- Ship the multilingual search beta on Friday.

## Action items
- Chen: finish the relevance evaluation by Wednesday.
- Luis: confirm the rollout owner and publish the dashboard link tomorrow.

## Open questions
- Whether Japanese indexing will remain in the beta.
"""

if os.getenv("ANTHROPIC_API_KEY"):
    notes = create_meeting_notes(transcript)
    notes_source = "Live Claude response"
elif transcript == DEMO_TRANSCRIPT:
    notes = DEMO_NOTES
    notes_source = "Offline demo notes (set ANTHROPIC_API_KEY to call Claude)"
else:
    notes = ""
    notes_source = "Claude call skipped; set ANTHROPIC_API_KEY to analyze this transcript."

print(notes_source)
if notes:
    print(notes)

Offline demo notes (set ANTHROPIC_API_KEY to call Claude)
## Summary
The team plans to ship the multilingual search beta on Friday.

## Decisions
- Ship the multilingual search beta on Friday.

## Action items
- Chen: finish the relevance evaluation by Wednesday.
- Luis: confirm the rollout owner and publish the dashboard link tomorrow.

## Open questions
- Whether Japanese indexing will remain in the beta.



## 5. Adapt the pipeline

Good next steps are speaker diarization, chunking long transcripts before the Claude call, storing source timestamps beside each action item, and adding a human review step before notes are published.

For production deployments, keep the FunASR endpoint private, validate file type and size before upload, apply authentication and rate limits at the service boundary, and follow your data-retention policy for both audio and transcripts.
